# Step 03 — Train HAR Head

Trains the classifier head on top of frozen backbone embeddings with all v2 improvements:
- **Focal Loss** (γ=2) — focuses gradient on hard/rare classes
- **WeightedRandomSampler** — balances mini-batches by class frequency
- **Mixup augmentation** — 3× virtual training samples in embedding space
- **CosineAnnealingLR** — smooth learning rate decay over 100 epochs
- **Optional SupCon pre-training** — contrastive pre-training before classification
- **Optional Bi-GRU head** — sequential model for temporal action discrimination

In [ ]:
import sys
from pathlib import Path
NB_DIR = Path.cwd()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

In [ ]:
# ── Training config ──────────────────────────────────────────────────────────
from lib.pipeline import PipelineConfig, step_train
from lib.constants import BACKBONE_VJEPA, BACKBONE_DINOV2

cfg = PipelineConfig(
    # Head
    head_arch            = 'mlp',    # 'mlp' | 'gru'
    train_epochs         = 100,
    # Loss & sampling
    use_focal_loss       = True,
    focal_gamma          = 2.0,
    use_weighted_sampler = True,
    use_mixup            = True,
    mixup_alpha          = 0.2,
    mixup_n_aug          = 2,
    # SupCon (set True for best results, adds ~50 epochs)
    use_supcon           = False,
    supcon_epochs        = 50,
    # Eval
    split_mode           = 'subject',
    backbones            = (BACKBONE_VJEPA, BACKBONE_DINOV2),
    skip_train_if_exists = False,
)
print(f"head={cfg.head_arch}  epochs={cfg.train_epochs}  focal={cfg.use_focal_loss}  mixup={cfg.use_mixup}  supcon={cfg.use_supcon}")

In [ ]:
train_results = step_train(cfg)
print("Training complete.")

In [ ]:
# Training curves
from lib.har_analysis import plot_training_history
from lib.paths import OUTPUTS_DIR

for backbone, stats in train_results.items():
    if stats.get('skipped'): continue
    hist = stats.get('history', [])
    if hist:
        plot_training_history(hist, OUTPUTS_DIR, tag=backbone)
        from IPython.display import Image, display
        display(Image(str(OUTPUTS_DIR / '04_training_history.png'), width=750))

In [ ]:
# Per-class results table
for backbone, stats in train_results.items():
    if stats.get('skipped'): continue
    r  = stats.get('val_report', {})
    si = stats.get('split_info', {})
    print(f"\n{'='*55}")
    print(f"  backbone    : {backbone}")
    print(f"  accuracy    : {r.get('accuracy', '?'):.3f}" if isinstance(r.get('accuracy'), float) else f"  accuracy: {r.get('accuracy')}")
    print(f"  macro_f1    : {r.get('macro avg',{}).get('f1-score','?'):.3f}" if isinstance(r.get('macro avg',{}).get('f1-score'), float) else "")
    print(f"  train/val   : {si.get('n_train_orig','?')} / {si.get('n_val','?')}")
    print(f"  aug samples : {si.get('n_train_aug','?')}")
    print(f"  best val_loss: {si.get('best_val_loss','?'):.4f}" if isinstance(si.get('best_val_loss'), float) else "")
    rows = []
    for cls, v in r.items():
        if isinstance(v, dict) and 'f1-score' in v:
            rows.append({'class': cls, 'F1': round(v['f1-score'],3),
                         'Precision': round(v['precision'],3),
                         'Recall': round(v['recall'],3),
                         'Support': int(v['support'])})
    if rows:
        df = pd.DataFrame(rows)
        df = df[df['class'] != 'accuracy']
        display(df.sort_values('F1'))

In [ ]:
# ── SupCon experiment ─────────────────────────────────────────────────────────
# Uncomment to run SupCon pre-training standalone and compare
#
# from lib.har_train import train_supcon, apply_supcon_projection, train_har_head
# import numpy as np
# from lib.paths import OUTPUTS_DIR
#
# npz = np.load(OUTPUTS_DIR / 'embeddings.npz', allow_pickle=True)
# X, y, cls = npz['X'].astype('float32'), npz['y'].astype('int64'), list(npz['class_names'])
# subj = npz['subjects'].astype(str)
#
# # Stage 1 — SupCon pre-training
# projector = train_supcon(X, y, emb_dim=X.shape[1], epochs=50)
# X_proj = apply_supcon_projection(projector, X)
# print(f'Projected shape: {X_proj.shape}')
#
# # Stage 2 — MLP on projected embeddings
# model, stats = train_har_head(
#     X_proj, y, cls,
#     exclude_labels=[], subjects=subj,
#     epochs=100, use_supcon=False,  # SupCon already done above
# )
# print(f"SupCon accuracy: {stats['val_report']['accuracy']:.3f}")